In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "customer360_dev.bronze.orders"
)

silver_df = spark.table(
    "customer360_dev.silver.orders"
)

reject_df = spark.table(
    "customer360_dev.quarantine.order_rejects"
)

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()
reject_count = reject_df.count()

print(f"Bronze rows   : {bronze_count:,}")
print(f"Silver rows   : {silver_count:,}")
print(f"Rejected rows : {reject_count:,}")

Bronze rows   : 5,000,126
Silver rows   : 4,740,000
Rejected rows : 260,126


In [0]:
customers_df = (
    spark.table("customer360_dev.silver.customers")
    .select("customer_id")
)

addresses_df = (
    spark.table("customer360_dev.silver.addresses")
    .select("address_id")
)

validated_df = (
    bronze_df
    .withColumn(
        "order_timestamp_cast",
        F.to_timestamp("order_timestamp")
    )
    .withColumn(
        "gross_amount_cast",
        F.col("gross_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "discount_amount_cast",
        F.col("discount_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "tax_amount_cast",
        F.col("tax_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "shipping_amount_cast",
        F.col("shipping_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "net_amount_cast",
        F.col("net_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "order_status_clean",
        F.upper(F.trim("order_status"))
    )
    .withColumn(
        "currency_clean",
        F.upper(F.trim("currency_code"))
    )
)

In [0]:
validated_df = (
    validated_df
    .join(
        customers_df.withColumn(
            "_customer_exists",
            F.lit(True)
        ),
        on="customer_id",
        how="left"
    )
)

In [0]:
shipping_addresses_df = (
    addresses_df
    .withColumnRenamed(
        "address_id",
        "shipping_address_id"
    )
    .withColumn(
        "_shipping_address_exists",
        F.lit(True)
    )
)

billing_addresses_df = (
    addresses_df
    .withColumnRenamed(
        "address_id",
        "billing_address_id"
    )
    .withColumn(
        "_billing_address_exists",
        F.lit(True)
    )
)

validated_df = (
    validated_df
    .join(
        shipping_addresses_df,
        on="shipping_address_id",
        how="left"
    )
    .join(
        billing_addresses_df,
        on="billing_address_id",
        how="left"
    )
)

Recreate DQ rules

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_missing_order_id",
        F.col("order_id").isNull()
        | (F.trim(F.col("order_id")) == "")
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_customer_not_found",
        F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_shipping_address_not_found",
        F.col("shipping_address_id").isNotNull()
        & F.col("_shipping_address_exists").isNull()
    )
    .withColumn(
        "dq_billing_address_not_found",
        F.col("billing_address_id").isNotNull()
        & F.col("_billing_address_exists").isNull()
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("order_status_clean").isin(
            "CREATED",
            "CONFIRMED",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED",
            "RETURNED"
        )
    )
    .withColumn(
        "dq_invalid_gross_amount",
        F.col("gross_amount_cast").isNull()
        | (F.col("gross_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_discount_amount",
        F.col("discount_amount_cast").isNull()
        | (F.col("discount_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_tax_amount",
        F.col("tax_amount_cast").isNull()
        | (F.col("tax_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_shipping_amount",
        F.col("shipping_amount_cast").isNull()
        | (F.col("shipping_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_net_amount",
        F.col("net_amount_cast").isNull()
        | (F.col("net_amount_cast") < 0)
    )
)

Recreate financial reconciliation

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "calculated_net_amount",
        (
            F.col("gross_amount_cast")
            - F.col("discount_amount_cast")
            + F.col("tax_amount_cast")
            + F.col("shipping_amount_cast")
        ).cast("decimal(14,2)")
    )
    .withColumn(
        "dq_amount_mismatch",
        F.abs(
            F.col("calculated_net_amount")
            - F.col("net_amount_cast")
        ) > F.lit(0.01)
    )
    .withColumn(
        "dq_invalid_order_timestamp",
        F.col("order_timestamp_cast").isNull()
    )
    .withColumn(
        "dq_invalid_currency",
        ~F.col("currency_clean").isin(
            "INR",
            "USD"
        )
    )
)

Critical failure flag

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_order_id")
    | F.col("dq_missing_customer_id")
    | F.col("dq_customer_not_found")
    | F.col("dq_shipping_address_not_found")
    | F.col("dq_billing_address_not_found")
    | F.col("dq_invalid_status")
    | F.col("dq_invalid_gross_amount")
    | F.col("dq_invalid_discount_amount")
    | F.col("dq_invalid_tax_amount")
    | F.col("dq_invalid_shipping_amount")
    | F.col("dq_invalid_net_amount")
    | F.col("dq_amount_mismatch")
    | F.col("dq_invalid_order_timestamp")
    | F.col("dq_invalid_currency")
)

Count valid candidates and duplicates removed

In [0]:
valid_candidate_count = (
    validated_df
    .filter(~F.col("dq_critical_failure"))
    .count()
)

duplicate_rows_removed = (
    valid_candidate_count - silver_count
)

print(
    f"Valid candidates before dedup: "
    f"{valid_candidate_count:,}"
)

print(
    f"Duplicate rows removed: "
    f"{duplicate_rows_removed:,}"
)

Valid candidates before dedup: 4,740,000
Duplicate rows removed: 0


Final reconciliation

In [0]:
accounted_for = (
    silver_count
    + reject_count
    + duplicate_rows_removed
)

difference = bronze_count - accounted_for

print(f"Bronze rows        : {bronze_count:,}")
print(f"Silver rows        : {silver_count:,}")
print(f"Rejected rows      : {reject_count:,}")
print(f"Duplicates removed : {duplicate_rows_removed:,}")
print(f"Accounted for      : {accounted_for:,}")
print(f"Difference         : {difference:,}")

Bronze rows        : 5,000,126
Silver rows        : 4,740,000
Rejected rows      : 260,126
Duplicates removed : 0
Accounted for      : 5,000,126
Difference         : 0


Produce PASS / FAIL result

In [0]:
result = [
    {
        "entity": "orders",
        "bronze_rows": bronze_count,
        "silver_rows": silver_count,
        "quarantine_rows": reject_count,
        "duplicates_removed": duplicate_rows_removed,
        "accounted_for_rows": accounted_for,
        "difference": difference,
        "status": "PASS" if difference == 0 else "FAIL"
    }
]

display(
    spark.createDataFrame(result)
)

accounted_for_rows,bronze_rows,difference,duplicates_removed,entity,quarantine_rows,silver_rows,status
5000126,5000126,0,0,orders,260126,4740000,PASS


Verify Silver referential integrity

In [0]:
orphan_customers = (
    silver_df
    .join(
        spark.table(
            "customer360_dev.silver.customers"
        ).select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "Orders with missing Silver customer:",
    orphan_customers
)

Orders with missing Silver customer: 0


Check warning metrics

In [0]:
warning_summary = (
    silver_df
    .select(
        F.sum(
            F.when(
                F.col(
                    "dq_warning_zero_value_delivered_order"
                ),
                1
            ).otherwise(0)
        ).alias("zero_value_delivered_orders"),

        F.sum(
            F.when(
                F.col(
                    "dq_warning_discount_exceeds_gross"
                ),
                1
            ).otherwise(0)
        ).alias("discount_exceeds_gross")
    )
)

display(warning_summary)

zero_value_delivered_orders,discount_exceeds_gross
0,0
